In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

{'commit': '', 'python': '3.13.16', 'torch': '2.14.1+cu130', 'gpu': 'Tesla T4', 'seed': 266}


In [2]:
src = open("/content/Part1/run_p1_rag.py").read()
def run_cell(n):
    block = src.split(f"# %% Cell {n} ")[1].split("\n# %%")[0]
    exec(block.split("\n", 1)[1], globals())
for n in range(1, 12):
    run_cell(n)
res.to_csv("review.csv", index=False)


{'commit': '', 'python': '3.13.16', 'torch': '2.14.1+cu130', 'gpu': 'Tesla T4', 'seed': 266}
trust levels: {'DOC1': 'high', 'DOC2': 'high', 'DOC3': 'medium'} | vintage years: {'DOC1': 2011, 'DOC2': 2014, 'DOC3': 2014}


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

using Qwen/Qwen2.5-VL-7B-Instruct
DOC1 Document.pdf pages: 37 | Appendix C starts on page: None
DOC2 Document2_Supplement_A_Outturns_2011-2013.pdf pages: 7 | Appendix C starts on page: None
DOC3 Document3_Briefing_B_Labour_Prices_Policy.pdf pages: 7 | Appendix C starts on page: 7
text chunks: 201 | figures/tables: 36
   doc_id  page               figure_id trust_level
0    DOC1     1               Table 1-1        high
1    DOC1     2              Figure 1-1        high
2    DOC1     3              Figure 1-2        high
3    DOC1     4              Figure 1-3        high
4    DOC1     5               Table 1-2        high
5    DOC1     5              Figure 1-4        high
6    DOC1     7               Table 1-3        high
7    DOC1     8              Figure 1-5        high
8    DOC1    10              Figure 1-6        high
9    DOC1    11               Table 1-4        high
10   DOC1    20              Figure 2-1        high
11   DOC1    21              Figure 2-2        high
12   

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

indexed 237 items


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13

40 answers in 992s | median 13.1s per question
status
answered            31
false_premise        3
session_override     3
conflict             2
not_found            1
label
normal           22
figure            7
false_premise     3
override          3
conflict          2
doc_followup      2
revision          1
22 rows with no figure refs
valid: 40 rows



In [4]:
from transformers import AutoProcessor
proc_hi = AutoProcessor.from_pretrained(VLM_NAME, min_pixels=256*28*28, max_pixels=1280*28*28)

@torch.no_grad()
def vlm_generate(system, user_text, images=(), max_new_tokens=300, hi=False):
    p = proc_hi if hi else proc
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": user_text}]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": content}]
    prompt = p.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = p(text=[prompt], images=list(images) or None, return_tensors="pt").to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return p.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

for n in (10, 11):
    run_cell(n)

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13

40 answers in 490s | median 8.9s per question
status
answered            32
conflict             3
session_override     3
false_premise        1
not_found            1
label
normal           22
figure            7
conflict          3
override          3
revision          2
doc_followup      2
false_premise     1
23 rows with no figure refs
valid: 40 rows



In [4]:
import os
from google.colab import files
print(os.getcwd(), os.path.exists("submission.csv"))
files.download("/content/submission.csv")
res.to_csv("review.csv", index=False)
files.download("review.csv")

/content True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>